# Etapa 5 — Clasificación Oficial (`STRICT`), Estadísticas y Reporte de Tesis

Replica `tools/classify_campaign_results.py` y el formato de `FINAL_CAMPAIGN_2026-10-27.md`:
1. Consolida todos los `summary.json` dentro de `campaign_workspace/`.
2. Clasifica cada ejecución en `qualified`, `requires_more_steps` o `incomplete` aplicando simultáneamente los umbrales oficiales de la tesis (`error_Tmax_abs_K < 0.05 K`, `nrmse_pct < 0.25 %`, `balance_pct < 0.5 %`).
3. Calcula agregados estadísticos por arquitectura, genera gráfico interactivo de comparación (Plotly), exporta `CAMPAIGN_CLASSIFICATION.json` + `CAMPAIGN_REPORT.md` y permite registrar las mejores corridas en `frontend/workspace/runs/` para verlas en Streamlit.

In [1]:
import json
import sys
import time
from pathlib import Path
import numpy as np

# Localiza la raíz PINN_V4 aunque el cuaderno se ejecute dentro de notebooks_campaign/
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks_campaign":
    ROOT = ROOT.parent
SRC = ROOT / "src"
for p in (SRC, ROOT):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

# Modo de ejecución:
# - 'demo_fast': subconjunto acotado de casos y pocos pasos para verificación rápida en minutos
# - 'full_campaign': reproduce todos los casos, semillas (0,1,2) y presupuestos largos de la tesis
RUN_MODE = "demo_fast"

CAMP_WS = ROOT / "notebooks_campaign" / "campaign_workspace"
FEM_CACHE = CAMP_WS / "fem_references"
CAMP_WS.mkdir(parents=True, exist_ok=True)
FEM_CACHE.mkdir(parents=True, exist_ok=True)

# Criterios oficiales de calificación de la campaña final de tesis (2026-10-27)
STRICT_CRITERIA = {"error_Tmax_abs_K": 0.05, "nrmse_pct": 0.25, "balance_pct": 0.5}
RELAXED_CRITERIA = {k: 2.0 * v for k, v in STRICT_CRITERIA.items()}

print(f"PINN_V4 raíz: {ROOT} | RUN_MODE={RUN_MODE!r} | Workspace campaña: {CAMP_WS}")

PINN_V4 raíz: C:\usr\ths_mia_fiis\tesis_MIA_2026-2_pinn | RUN_MODE='demo_fast' | Workspace campaña: C:\usr\ths_mia_fiis\tesis_MIA_2026-2_pinn\notebooks_campaign\campaign_workspace


In [2]:
import pinn_thermal.models  # noqa: F401
from pinn_thermal.core.artifact import SolutionArtifact
from pinn_thermal.core.manifest import write_json_atomic
from pinn_thermal.core.problem import CaseCatalog
from pinn_thermal.core.trainer import PINNTrainer, TrainConfig
from pinn_thermal.fem.runner import FEniCSxAdapter

catalog = CaseCatalog(ROOT / "cases")

def load_or_solve_reference(case_id: str, problem, use_fem: bool = True):
    """Obtiene referencia analítica exacta si existe, o carga/resuelve caché FEM P2 en WSL."""
    xy, _, _ = problem.evaluation()
    exact = problem.reference(xy)
    if exact is not None and np.isfinite(exact).all():
        return np.asarray(exact, dtype=float), "analytic_reference"

    fem_dir = FEM_CACHE / case_id
    fem_json = fem_dir / "fem_l1.json"
    if fem_json.is_file() and fem_json.with_suffix(".npz").is_file():
        art = SolutionArtifact.load(fem_json)
        return art.predict(xy)["T"], f"FEM_WSL_cache:{case_id}"

    if use_fem and FEniCSxAdapter().is_available().get("available"):
        art = FEniCSxAdapter().solve(problem, output_dir=fem_dir, level=1)
        return art.predict(xy)["T"], f"FEM_WSL_L1:{case_id}"

    return None, "physics_only_no_reference"

def extract_metrics(result: dict) -> dict:
    ev = result.get("evaluation") or {}
    bal = ev.get("balance") or {}
    return {
        "error_Tmax_abs_K": ev.get("error_Tmax_abs_K"),
        "nrmse_pct": ev.get("nrmse_pct"),
        "balance_pct": bal.get("global_derived_pct"),
        "Tmax_pred_C": ev.get("Tmax_pred_C"),
        "elapsed_s": result.get("elapsed_s"),
    }
from collections import Counter
from pinn_thermal.visualization import GridSampler, InteractivePlotter
from frontend.state import WorkspaceStore

def extract_summary_metrics(record: dict):
    if not isinstance(record, dict):
        return None
    direct = {k: record.get(k) for k in STRICT_CRITERIA}
    if all(v is not None for v in direct.values()):
        return direct
    stages = record.get("stages") or []
    for st in reversed(stages):
        m = st.get("metrics") or {}
        if all(m.get(k) is not None for k in STRICT_CRITERIA):
            return {k: m[k] for k in STRICT_CRITERIA}
    return None

rows = []
for s_file in sorted(CAMP_WS.rglob("*.json")):
    if s_file.name in ("promoted_models.json", "CAMPAIGN_CLASSIFICATION.json", "stage0_mms_audit.json", "stage0_fem_cache.json"):
        continue
    data = json.loads(s_file.read_text(encoding="utf-8-sig"))
    if not isinstance(data, dict):
        continue
    for key, rec in data.items():
        m = extract_summary_metrics(rec)
        if m is None:
            continue
        qual = all(float(m[k]) < STRICT_CRITERIA[k] for k in STRICT_CRITERIA)
        rows.append({
            "source": str(s_file.relative_to(CAMP_WS)),
            "key": key,
            "case": rec.get("case", str(key).split(":")[0]),
            "architecture": rec.get("architecture", ""),
            "error_Tmax_abs_K": float(m["error_Tmax_abs_K"]),
            "nrmse_pct": float(m["nrmse_pct"]),
            "balance_pct": float(m["balance_pct"]),
            "qualification": "qualified" if qual else "requires_more_steps",
        })

counts = Counter(r["qualification"] for r in rows)
write_json_atomic(
    CAMP_WS / "CAMPAIGN_CLASSIFICATION.json",
    {"criteria": STRICT_CRITERIA, "total": len(rows), "counts": dict(counts), "rows": rows},
)
print(f"Total clasificadas: {len(rows)} | Conteo: {dict(counts)}")

# Estadísticas por arquitectura
by_arch = {}
for r in rows:
    by_arch.setdefault(r["architecture"], []).append(r)

lines = [
    "# Reporte Consolidado de Campaña Experimental — `PINN_V4`",
    "",
    f"- **Modo de ejecución**: `{RUN_MODE}`",
    f"- **Total de ejecuciones evaluadas**: {len(rows)} (`qualified`: **{counts['qualified']}**, `requires_more_steps`: **{counts['requires_more_steps']}**)",
    "",
    "## Resumen por Arquitectura",
    "",
    "| Arquitectura | Corridas | Calificadas | NRMSE medio (%) | Error Tmax medio (K) | Balance medio (%) |",
    "|---|---:|---:|---:|---:|---:|",
]
for arch, items in sorted(by_arch.items()):
    q_cnt = sum(1 for i in items if i["qualification"] == "qualified")
    lines.append(
        f"| `{arch}` | {len(items)} | {q_cnt} | "
        f"{np.mean([i['nrmse_pct'] for i in items]):.4f} | "
        f"{np.mean([i['error_Tmax_abs_K'] for i in items]):.4f} | "
        f"{np.mean([i['balance_pct'] for i in items]):.4f} |"
    )

report_path = CAMP_WS / "CAMPAIGN_REPORT.md"
report_path.write_text("\n".join(lines) + "\n", encoding="utf-8")
print("\n".join(lines))

Total clasificadas: 42 | Conteo: {'requires_more_steps': 41, 'qualified': 1}
# Reporte Consolidado de Campaña Experimental — `PINN_V4`

- **Modo de ejecución**: `demo_fast`
- **Total de ejecuciones evaluadas**: 42 (`qualified`: **1**, `requires_more_steps`: **41**)

## Resumen por Arquitectura

| Arquitectura | Corridas | Calificadas | NRMSE medio (%) | Error Tmax medio (K) | Balance medio (%) |
|---|---:|---:|---:|---:|---:|
| `CUSPR` | 8 | 0 | 17.4487 | 46.7976 | 628.5819 |
| `M0` | 4 | 1 | 14.0438 | 60.0284 | 6207.8894 |
| `M0R` | 2 | 0 | 13.9350 | 59.0941 | 51.9522 |
| `M1` | 12 | 0 | 16.6757 | 51.9431 | 93.6437 |
| `M1_EXACT` | 12 | 0 | 12.0385 | 49.2906 | 58.0803 |
| `M2` | 4 | 0 | 31.4902 | 69.3825 | 36.8951 |


In [3]:
# Publicar opcionalmente una mejor corrida hacia el workspace compartido con Streamlit
store = WorkspaceStore(ROOT / "frontend" / "workspace", ROOT / "cases")
long_cand = CAMP_WS / "stage4_long_focused" / "mms_sine"
if long_cand.is_dir():
    subdirs = [d for d in sorted(long_cand.iterdir()) if (d / "model.pt").is_file()]
    if subdirs:
        best_art = SolutionArtifact.load(subdirs[0])
        published = store.save_artifact(best_art, run_name=f"campaign_best__{best_art.problem.id}__{best_art.architecture}")
        print("Mejor artefacto publicado para Streamlit/Jupyter en:", published)

Mejor artefacto publicado para Streamlit/Jupyter en: C:\usr\ths_mia_fiis\tesis_MIA_2026-2_pinn\frontend\workspace\runs\campaign_best__mms_sine__M1
